# Очистка данных

Продолжение [проверки качества данных](01_data_quality.ipynb). Фиксирую, какие правила очистки беру из проверки и какие допущения при этом делаю, затем по каждому правилу смотрю, сколько строк уходит. В конце сохраняю очищенную таблицу `events_clean` — с ней работают следующие ноутбуки. Исходная таблица `events` не меняется.

In [1]:
import duckdb
from pathlib import Path

# корень проекта: ноутбук может запускаться и из notebooks/, и из корня
ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent

# read_only: базу можно читать одновременно из ноутбука и из терминала
con = duckdb.connect(str(ROOT / 'data/processed/shop.duckdb'), read_only=True)

def q(sql):
    """Выполнить SQL и показать результат таблицей."""
    return con.sql(sql).df()

## Правила очистки и допущения

Что выяснила на [проверке качества](01_data_quality.ipynb) и как это использую при очистке. Исходную таблицу `events` не меняю, очищенные данные собираю отдельно.

| # | Правило | Почему (из проверки) | Допущение | Чем рискую |
|---|---|---|---|---|
| 1 | Убираю точные дубли: одинаковая строка остаётся один раз | 408 960 лишних строк (5,24%), из них 373 297 (91%) — `remove_from_cart`, 35 144 — `cart`, в покупках и просмотрах почти нет (302 и 217). Копии совпадают до секунды, бывает до 29 копий одного события, и дубли есть во все 62 дня | это постоянная ошибка записи, а не действия человека: так быстро и столько раз подряд человек не кликает | если удаление нескольких штук товара записывается несколькими одинаковыми событиями, теряю количество; количество не анализирую. Воронку по пользователям дубли не меняют (704 985, 159 572 и 49 861 с дублями и без) |
| 2 | Пропуски бренда восстанавливаю по тому же товару | у 1 792 товаров бренд то указан, то нет — это случайные пропуски при записи | бренд товара не меняется, поэтому его можно взять из других событий этого товара | ошибусь, если бренд товара действительно менялся; такие товары (их 21) разбирает правило 3 |
| 3 | Для 21 товара с двумя брендами беру последний бренд | на этих товарах `candy` встречается только до 13 декабря, потом его сменяет другой бренд; при этом у `candy` есть свои товары до 22 января | это была ошибка каталога, которую потом исправили; истории каталога нет, поэтому точно не знаю | 96 событий с `candy` из 7,8 млн; если допущение неверно, ошибка будет только в анализе по брендам |
| 4 | Остальные пропуски бренда и `category_code` не заполняю; категории сравниваю по `category_id` | у большинства товаров бренд либо есть всегда, либо нет никогда — это свойство товара в каталоге; `category_code` пуст в 98% строк, `category_id` заполнен полностью | отличить no-name товары от брендовых с незаполненным брендом нельзя | выводы по брендам — только с оговоркой: долю бренда в выручке, рейтинг по суммам и сравнение «брендовые против небрендовых» не считаю |
| 5 | Окно анализа — с 16 декабря, но саму таблицу по дате не обрезаю: 1–15 декабря остаются историей | покупок без корзины 9,67%, в первые дни декабря около 30%, со второй половины декабря доля держится на 7–9%: товары клали в корзину в ноябре, которого нет в данных | после середины декабря покупки без корзины — уже не обрезка данных. В 66% таких случаев человек в той же сессии клал в корзину другой товар, поэтому оставшиеся 7–9% — скорее потеря событий или кнопка «купить сразу» | если обрезать саму таблицу, проблема повторяется: корзины 1–15 декабря пропадают, и в первую неделю «без корзины» снова около 19% (проверка ниже). Поэтому покупки и шаги воронки беру с 16 декабря, а корзины ищу по всей истории |
| 6 | Деньги (выручка, средняя и медианная цена) считаю только по `price > 0` | 57 событий с отрицательной ценой у 5 товаров (54 покупки, цена у товара всегда одна — похоже на скидку, возврат или подарочную карту); 15 212 событий с нулевой ценой, почти все в просмотрах и корзине, покупка за 0 одна | нулевая цена — возможно, подарок по акции; точно не знаю | в воронке эти события оставляю: действие пользователя было, цена в воронке не участвует |
| 7 | 128 сессий с несколькими `user_id` не удаляю, а помечаю флагом `is_multi_user_session` | 929 событий (0,012%); ID меняются через секунды и идут по порядку — на разных людей не похоже. Но у части ID есть другие сессии, у одного 84 | в основном это новые ID одному посетителю (сайт или браузер выдавал новый ID), хотя часть может быть настоящими разными аккаунтами — разделить по данным нельзя | если это один человек, он считается как несколько пользователей и занижает конверсию и удержание; поэтому в анализе по сессиям исключаю, а воронку и когорты считаю с ними и без и сравниваю. Для A/B-теста это риск: при смене ID один человек может попасть в обе группы |
| 8 | События без `user_session` оставляю, в анализе по сессиям они выпадают | 2 093 события (0,03%), случайные пропуски | в остальном это обычные события | если пропуск сессии связан с поведением, анализ по сессиям сместится; в анализе по пользователям эти события учитываются |
| 9 | Спад 31 декабря, субботы и всплеск 27–28 января оставляю | в канун Нового года события есть в каждый час — не похоже на сбой; в субботу медиана событий на 22% ниже, чем во вторник; причина всплеска 27–28 января неизвестна, дубли объясняют только малую часть | спад и субботы — обычное поведение; всплеск считаю настоящим трафиком, пока не найден сбой | проверяю, что выводы по воронке не меняются без 27–28 января; A/B-тест держу полными неделями |

**Общие допущения для анализа**
- Один `user_id` — один человек. Если один человек заходит с разных устройств и получает разные ID, пользователей будет больше, а конверсия — немного ниже реальной.
- Воронку считаю по людям, которые дошли до каждого шага; порядок событий не проверяю. Покупки без корзины учитываю как покупку.
- «Новый пользователь» для когорт — тот, кто впервые появился в данных. Часть «новых» в декабре на самом деле приходила в ноябре.
- Данные заканчиваются 31 января: пользователи конца января не успели купить или вернуться, поэтому их конверсия и удержание занижены. Для когорт беру только те, у которых было время вернуться.

## Как собрана `events_clean`

Запрос — в [`sql/02_clean.sql`](../sql/02_clean.sql), за один проход:
1. `SELECT DISTINCT *` — убираю точные дубли;
2. для каждого товара беру последний известный бренд (`ARG_MAX(brand, event_time)`) — это закрывает и случайные пропуски бренда, и ошибку с `candy`;
3. добавляю флаг `is_multi_user_session` для сессий с несколькими `user_id`;
4. по дате таблицу не обрезаю — окно анализа задаю в запросах.

## Что изменилось: до и после очистки

In [2]:
q("""
SELECT 'events' AS tbl,
       COUNT(*)                                                   AS rows,
       COUNT(DISTINCT user_id)                                    AS users,
       SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END)   AS purchases,
       COUNT(*) - COUNT(brand)                                    AS brand_missing,
       SUM(CASE WHEN brand = 'candy' THEN 1 ELSE 0 END)           AS candy_events,
       NULL                                                       AS flagged_rows
FROM events
UNION ALL
SELECT 'events_clean',
       COUNT(*),
       COUNT(DISTINCT user_id),
       SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END),
       COUNT(*) - COUNT(brand),
       SUM(CASE WHEN brand = 'candy' THEN 1 ELSE 0 END),
       SUM(CASE WHEN is_multi_user_session THEN 1 ELSE 0 END)
FROM events_clean
""")

,tbl,rows,users,purchases,brand_missing,candy_events,flagged_rows
0,events,7798038,725736,476973.0,3285919,203.0,NaN
1,events_clean,7389078,725736,476671.0,3097321,1551.0,915.0


Проверяю, что правила сработали: дублей нет, у каждого товара не больше одного бренда, флаг стоит на тех самых сессиях.

In [3]:
q("""
SELECT
    (SELECT COUNT(*) FROM (SELECT *, COUNT(*) AS copies FROM events_clean
                           GROUP BY ALL HAVING copies > 1))              AS duplicated_events,
    (SELECT COUNT(*) FROM (SELECT product_id FROM events_clean WHERE brand IS NOT NULL
                           GROUP BY product_id HAVING COUNT(DISTINCT brand) > 1)) AS multi_brand_products,
    (SELECT COUNT(DISTINCT user_session) FROM events_clean
     WHERE is_multi_user_session)                                         AS flagged_sessions,
    (SELECT COUNT(*) FROM events_clean WHERE price <= 0)                  AS price_not_positive
""")

,duplicated_events,multi_brand_products,flagged_sessions,price_not_positive
0,0,0,128,15087


## Почему таблицу не обрезаю по дате

Считаю долю покупок без корзины по неделям двумя способами: корзины ищу по всей истории и только с 16 декабря.

In [4]:
q("""
WITH purchases AS (
    SELECT user_id, product_id, MIN(event_time) AS first_purchase
    FROM events_clean
    WHERE event_type = 'purchase'
    GROUP BY user_id, product_id
),
carts_all AS (
    SELECT DISTINCT user_id, product_id FROM events_clean
    WHERE event_type = 'cart'
),
carts_from_16 AS (
    SELECT DISTINCT user_id, product_id FROM events_clean
    WHERE event_type = 'cart' AND event_time >= '2019-12-16'
)
SELECT
    DATE_TRUNC('week', p.first_purchase)::date AS week,
    COUNT(*) AS purchased_pairs,
    ROUND(100.0 * SUM(CASE WHEN a.user_id IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS no_cart_pct_all_history,
    ROUND(100.0 * SUM(CASE WHEN f.user_id IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS no_cart_pct_cut_at_16
FROM purchases p
LEFT JOIN carts_all a     ON a.user_id = p.user_id AND a.product_id = p.product_id
LEFT JOIN carts_from_16 f ON f.user_id = p.user_id AND f.product_id = p.product_id
WHERE p.first_purchase >= '2019-12-16'
GROUP BY week
ORDER BY week
""")

,week,purchased_pairs,no_cart_pct_all_history,no_cart_pct_cut_at_16
0,2019-12-16,49132,8.66,19.28
1,2019-12-23,37974,8.16,11.54
2,2019-12-30,30907,7.16,9.21
3,2020-01-06,52331,7.30,8.84
4,2020-01-13,61132,8.71,9.71
5,2020-01-20,57187,8.79,9.44
6,2020-01-27,55255,7.32,7.83


### Вывод по очистке

- **Дубли убраны:** 408 960 строк (5,24%). Пользователей и покупателей в воронке столько же, меняется только число событий, в основном удалений из корзины.
- **Бренд восстановлен там, где он известен у того же товара:** пропусков бренда 3 097 321 вместо 3 285 919, товаров с двумя брендами не осталось. Событий с `candy` стало больше (1 551 вместо 203), и это ожидаемо: на 21 «двухбрендовом» товаре `candy` заменён последним брендом, а у 15 собственных товаров `candy` заполнились случайные пропуски бренда.
- **128 сессий с несколькими `user_id` не удалены, а помечены флагом** — дальше считаю воронку и когорты с ними и без и сравниваю.
- **Цены не трогаю:** события с `price <= 0` остаются в таблице, деньги считаю только по `price > 0`.
- **Таблицу по дате не обрезаю.** Если обрезать с 16 декабря, в первую неделю покупок без корзины снова около 19% вместо 8–9%: корзины 1–15 декабря пропадают, и обрезка ноября повторяется. Поэтому первые две недели остаются историей, а окно анализа (покупки, шаги воронки) — с 16 декабря.